# 03: CNN + LSTM (LRCN) Baseline
**Group 11 · COSE474 Deep Learning · Korea University · Spring 2026**

## Overview

This notebook trains the **LRCN baseline** — VGG16 (frozen) as a per-frame feature extractor with an LSTM on top to model temporal dynamics across 48 frames.

**Why LRCN?**
Notebook 02 showed that a single-frame CNN achieves 13.57% val accuracy on KSL-77. Signs are movements — the same hand shape can mean different signs depending on how it moves. LRCN (Donahue et al. 2015) adds an LSTM to learn the temporal pattern across frames, which is the missing piece.

**Architecture:**
```
(B, 32, 3, 224, 224)   <- clip of 48 frames
         |
VGG16 conv (FROZEN, ImageNet)  applied per-frame  ->  (B*32, 512, 7, 7)
         |
AdaptiveAvgPool + Flatten                          ->  (B*32, 25088)
         |
Linear(25088 -> 512) + BatchNorm + ReLU + Dropout ->  (B*32, 512)
         |
Reshape  ->  (B, 32, 512)
         |
LSTM (hidden=256, 1 layer)                         ->  final hidden state (B, 256)
         |
Dropout -> Linear(256, NUM_CLASSES_ACTUAL)         ->  (B, 67)
```

**Key changes from previous version:**
- NUM_FRAMES: 16 -> 32 (matches notebook 01/02)
- NUM_CLASSES: 77 -> 67 (dynamic — 10 classes missing from dataset)
- KSL word labels on confusion matrix axes
- Dynamic LABEL_MAP and INDEX_TO_WORD
- Local paths (no Google Drive)

**Outputs:**
```
models/checkpoints/lrcn.pth
results/figures/03_lrcn_curves.png
results/figures/03_lrcn_confusion.png
results/logs/03_lrcn_log.csv
```

**Compare against:** Notebook 02 baseline CNN = 13.57%

# Setup & Config

In [1]:
# Author: Marcello Nico Valerian
# Notebook 03 — LRCN (CNN + LSTM) baseline
# Updated: 32 frames, 67 classes, word labels on confusion matrix

import os
import csv
import random
import datetime
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from PIL import Image
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ── Paths (local — no Google Drive) ─────────────────────────
BASE_DIR   = Path('.')
FRAMES_DIR = BASE_DIR / 'frames'
CKPT_DIR   = BASE_DIR / 'models' / 'checkpoints'
FIGS_DIR   = BASE_DIR / 'results' / 'figures'
LOGS_DIR   = BASE_DIR / 'results' / 'logs'
for d in [CKPT_DIR, FIGS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CKPT_PATH = str(CKPT_DIR / 'lrcn.pth')

# ── Hyperparameters ──────────────────────────────────────────
NUM_FRAMES     = 32
IMG_SIZE       = 224
BATCH          = 8        # memory constraint: 48 frames per clip
LEARNING_RATE  = 1e-4     # lower than CNN baseline — LSTM training is more sensitive
WEIGHT_DECAY   = 1e-4
NUM_EPOCHS     = 50
PATIENCE       = 10
LSTM_HIDDEN    = 64
LSTM_LAYERS    = 1
DROPOUT        = 0.4
RANDOM_SEED    = 42
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD  = [0.229, 0.224, 0.225]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"NUM_FRAMES : {NUM_FRAMES}")
print(f"BATCH      : {BATCH}")

Device : cuda
GPU    : NVIDIA A100 80GB PCIe MIG 2g.20gb
NUM_FRAMES : 32
BATCH      : 8


# KSL-77 Word Labels
Same dictionary as notebooks 01 and 02. Used for confusion matrix axes.

In [2]:
KSL_WORDS = {
     1: 'hi',           2: 'what',          3: 'meat',          4: 'bi bim rice',
     5: 'glad',         6: 'hobby',         7: 'me',            8: 'movie',
     9: 'face',        10: 'see',          11: 'name',         12: 'read',
    13: 'thank',       14: 'equal',        15: 'sorry',        16: 'eat',
    17: 'fine',        18: 'do effort',    19: 'next',         20: 'age',
    21: 'again',       22: 'how many',     23: 'day',          24: 'good, nice',
    25: 'when',        26: 'we',           27: 'subway',       28: 'be friendly',
    29: 'bus',         30: 'ride',         31: 'cell phone',   32: 'where',
    33: 'number',      34: 'location',     35: 'guide',        36: 'responsibility',
    37: 'who',         38: 'arrive',       39: 'family',       40: 'time',
    41: 'friend',      42: 'help',         43: 'know',         44: 'work',
    45: 'parents',     46: '10 minutes',   47: 'walk',         48: 'feel',
    49: 'think',       50: 'money',        51: 'teach',        52: 'meet',
    53: 'education',   54: 'want',         55: 'rest',         56: 'talk',
    57: 'hospital',    58: 'go',           59: 'house',        60: 'school',
    61: 'like',        62: 'bad',          63: 'together',     64: 'because',
    65: 'no',          66: 'now',          67: 'different',    68: 'tired',
    69: 'yes',         70: 'do',           71: 'strange',      72: 'difficult',
    73: 'visit',       74: 'need',         75: 'far',          76: 'person',
    77: 'much',
}
print(f"KSL words loaded: {len(KSL_WORDS)}")

KSL words loaded: 77


# Dataset

`KSLDataset` returns the full `(NUM_FRAMES, C, H, W)` clip — identical to notebook 01.

Uses `label_map` for dynamic remapping: the 10 missing classes are excluded,
so labels are contiguous 0..66 rather than 0..76 with gaps.

In [3]:
class KSLDataset(Dataset):
    """
    Reads NUM_FRAMES JPGs per clip.
    Returns (NUM_FRAMES, C, H, W) tensor + remapped 0-based label.
    """
    def __init__(self, clip_dirs, transform=None,
                 label_map=None, num_frames=NUM_FRAMES):
        self.clips      = clip_dirs
        self.transform  = transform
        self.label_map  = label_map
        self.num_frames = num_frames

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_path   = self.clips[idx]
        folder_name = os.path.basename(clip_path)
        class_id    = int(folder_name.split('_')[1])
        label = self.label_map[class_id] if self.label_map is not None \
                else class_id - 1

        frame_files = sorted(os.listdir(clip_path))
        if len(frame_files) < self.num_frames:
            frame_files += [frame_files[-1]] * (self.num_frames - len(frame_files))
        frame_files = frame_files[:self.num_frames]

        tensors = []
        for fname in frame_files:
            img = Image.open(os.path.join(clip_path, fname)).convert('RGB')
            if self.transform:
                img = self.transform(img)
            tensors.append(img)

        return torch.stack(tensors), label   # (NUM_FRAMES, C, H, W)

# Label Map + Signer-based Train/Val Split

Dynamic label remapping handles the 10 missing classes.
Signer split: 00-15 train, 16-19 val — identical to notebooks 01/02.

In [4]:
TRAIN_SIGNERS = {f"{i:02d}" for i in range(16)}
VAL_SIGNERS   = {f"{i:02d}" for i in range(16, 20)}

all_clips = sorted([
    str(FRAMES_DIR / d)
    for d in os.listdir(FRAMES_DIR)
    if (FRAMES_DIR / d).is_dir()
])

# build label map from present classes only
present_class_ids  = sorted({int(os.path.basename(c).split('_')[1])
                              for c in all_clips})
missing_class_ids  = sorted(set(KSL_WORDS.keys()) - set(present_class_ids))
LABEL_MAP          = {cid: i for i, cid in enumerate(present_class_ids)}
NUM_CLASSES_ACTUAL = len(present_class_ids)
INDEX_TO_WORD      = [KSL_WORDS[cid] for cid in present_class_ids]

train_clips = [c for c in all_clips
               if os.path.basename(c).split('_')[0] in TRAIN_SIGNERS]
val_clips   = [c for c in all_clips
               if os.path.basename(c).split('_')[0] in VAL_SIGNERS]

print(f"Total clips        : {len(all_clips)}")
print(f"Classes present    : {NUM_CLASSES_ACTUAL}")
print(f"Missing class IDs  : {missing_class_ids}")
print(f"Missing words      : {[KSL_WORDS[c] for c in missing_class_ids]}")
print(f"Train clips        : {len(train_clips)}  (signers 00-15)")
print(f"Val clips          : {len(val_clips)}   (signers 16-19)")

Total clips        : 1229
Classes present    : 67
Missing class IDs  : [12, 19, 28, 33, 35, 45, 46, 53, 73, 75]
Missing words      : ['read', 'next', 'be friendly', 'number', 'guide', 'parents', '10 minutes', 'education', 'visit', 'far']
Train clips        : 971  (signers 00-15)
Val clips          : 258   (signers 16-19)


# Transforms and DataLoaders

No augmentation — that is notebook 04's job.
Batch size = 8: each sample is 32 frames x 3 x 224 x 224 — memory constraint.

In [5]:
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

train_dataset = KSLDataset(train_clips, base_transform,
                           label_map=LABEL_MAP, num_frames=NUM_FRAMES)
val_dataset   = KSLDataset(val_clips,   base_transform,
                           label_map=LABEL_MAP, num_frames=NUM_FRAMES)

train_loader = DataLoader(train_dataset, batch_size=BATCH,
                          shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH,
                          shuffle=False, num_workers=4, pin_memory=True)

print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")
print(f"Batch size    : {BATCH}")

# sanity check
clip, lbl = train_dataset[0]
print(f"Clip shape    : {clip.shape}")   # (32, 3, 224, 224)
print(f"Sample label  : {lbl} -> '{INDEX_TO_WORD[lbl]}'")

Train samples : 971
Val samples   : 258
Batch size    : 8
Clip shape    : torch.Size([32, 3, 224, 224])
Sample label  : 0 -> 'hi'


# Model — LRCN

**Forward pass:**
1. Reshape `(B, T, C, H, W) -> (B*T, C, H, W)` — all frames through VGG16 in one batched call
2. VGG16 conv + avgpool -> flatten to `(B*T, 25088)`
3. `Linear(25088, 512)` + BatchNorm + ReLU + Dropout — reduces dim before LSTM
4. Reshape back to `(B, T, 512)` — restore temporal sequence
5. LSTM reads sequence -> final hidden state `h_T` (summary of full clip)
6. Dropout -> `Linear(256, NUM_CLASSES_ACTUAL)` -> class logits

**Note:** `num_classes` defaults to `NUM_CLASSES_ACTUAL` (67) not the config value (77).

In [10]:
class LRCN(nn.Module):
    """
    VGG16 (frozen, ImageNet) -> per-frame feature -> LSTM -> classifier.
    num_classes defaults to NUM_CLASSES_ACTUAL (67, dynamic from dataset).
    """
    def __init__(self,
                 num_classes  = None,
                 feature_dim  = 512,
                 lstm_hidden  = LSTM_HIDDEN,
                 lstm_layers  = LSTM_LAYERS,
                 dropout      = DROPOUT):
        super(LRCN, self).__init__()
        if num_classes is None:
            num_classes = NUM_CLASSES_ACTUAL

        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.features = vgg.features
        self.avgpool  = vgg.avgpool

        for p in self.features.parameters():
            p.requires_grad = False

        self.frame_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            
        )

        self.lstm = nn.LSTM(
            input_size  = feature_dim,
            hidden_size = LSTM_HIDDEN,
            num_layers  = lstm_layers,
            batch_first = True,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(LSTM_HIDDEN, num_classes),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        with torch.no_grad():
            x = self.features(x)
            x = self.avgpool(x)
        x = self.frame_fc(x)            # (B*T, 512)
        x = x.view(B, T, -1)           # (B, T, 512)
        out, (h_n, _) = self.lstm(x)
        last = out[:, -1, :]           # final time-step
        return self.classifier(last)   # (B, num_classes)


model = LRCN().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Output classes   : {NUM_CLASSES_ACTUAL}")
print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {total - trainable:,}")

Output classes   : 67
Trainable params : 12,998,915
Frozen params    : 14,714,688


# Training Setup

- Loss: CrossEntropyLoss
- Optimizer: Adam on trainable params only (frame_fc + LSTM + classifier)
- LR: 1e-4 (lower than CNN baseline — LSTM training is more sensitive)
- Scheduler: StepLR — halve every 10 epochs
- Early stopping: patience = 10

In [11]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print(f"Loss       : CrossEntropyLoss")
print(f"Optimizer  : Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler  : StepLR (step=10, gamma=0.5)")
print(f"Max epochs : {NUM_EPOCHS}")
print(f"Patience   : {PATIENCE}")

Loss       : CrossEntropyLoss
Optimizer  : Adam (lr=0.0001, wd=0.0001)
Scheduler  : StepLR (step=10, gamma=0.5)
Max epochs : 50
Patience   : 10


# Train and Validation Functions

In [12]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for clips, labels in loader:
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(clips)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += logits.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for clips, labels in loader:
            clips, labels = clips.to(device), labels.to(device)
            logits = model(clips)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            preds       = logits.argmax(1)
            correct    += preds.eq(labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), 100.0 * correct / total, all_preds, all_labels

## Training Loop

**Compare against:** Notebook 02 baseline CNN = 13.57%

Each epoch is slower than notebook 02 — every sample passes 32 frames through VGG16.

In [ ]:
history = {k: [] for k in ['train_loss','train_acc','val_loss','val_acc']}
best_val_acc, patience_count = 0.0, 0
best_preds, best_labels      = [], []

print("Starting training (LRCN baseline)...\n")
print(f"{'Epoch':>5} {'Tr Loss':>9} {'Tr Acc':>8} {'Vl Loss':>9} {'Vl Acc':>8}")
print('─' * 48)

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    vl_loss, vl_acc, vl_preds, vl_labels = val_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    for k, v in zip(['train_loss','train_acc','val_loss','val_acc'],
                    [tr_loss, tr_acc, vl_loss, vl_acc]):
        history[k].append(v)

    marker = ' <-- best' if vl_acc > best_val_acc else ''
    print(f"{epoch:>5} {tr_loss:>9.4f} {tr_acc:>7.2f}% {vl_loss:>9.4f} {vl_acc:>7.2f}%{marker}")

    if vl_acc > best_val_acc:
        best_val_acc   = vl_acc
        best_preds     = vl_preds
        best_labels    = vl_labels
        patience_count = 0
        torch.save(model.state_dict(), CKPT_PATH)
    else:
        patience_count += 1

    if patience_count >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\nBest val accuracy   : {best_val_acc:.2f}%")
print(f"CNN baseline (nb02) : 13.57%")
print(f"Delta from baseline : {best_val_acc - 13.57:+.2f} pp")
print(f"Checkpoint          : {CKPT_PATH}")

Starting training (LRCN baseline)...

Epoch   Tr Loss   Tr Acc   Vl Loss   Vl Acc
────────────────────────────────────────────────
    1    4.2601    1.44%    4.1970    2.33% <-- best
    2    4.2080    1.75%    4.1843    3.88% <-- best
    3    4.1751    2.88%    4.1737    3.88%
    4    4.1479    4.43%    4.1631    4.26% <-- best
    5    4.1246    4.12%    4.1587    5.43% <-- best
    6    4.1061    7.21%    4.1463    4.65%
    7    4.0686    7.52%    4.1305    4.26%
    8    4.0318    9.17%    4.1098    5.81% <-- best
    9    3.9849   11.12%    4.0929    6.20% <-- best
   10    3.9423   13.29%    4.0779    6.98% <-- best
   11    3.8616   16.58%    4.0595    6.98%
   12    3.7917   19.57%    4.0495    7.75% <-- best
   13    3.7026   25.23%    4.0218    6.59%
   14    3.6406   26.78%    4.0029    7.36%
   15    3.5934   29.56%    3.9930    7.75%
   16    3.4995   31.62%    3.9676    9.69% <-- best
   17    3.4516   36.66%    3.9538    9.30%
   18    3.3917   38.41%    3.9426    9.

# Results
## Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('LRCN — Loss'); ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.axhline(y=13.57, color='gray', linestyle='--', label='CNN baseline 13.57%')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('LRCN — Accuracy'); ax2.legend()

plt.tight_layout()
path = str(FIGS_DIR / '03_lrcn_curves.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

## Confusion matrix

Axes show actual KSL word names instead of class IDs.
Compare against `02_baseline_cnn_confusion.png` — the LSTM should
resolve specific class pairs that share a hand shape but differ in motion.

In [ ]:
cm = confusion_matrix(
    best_labels, best_preds,
    labels=list(range(NUM_CLASSES_ACTUAL))
)

fig, ax = plt.subplots(figsize=(22, 20))
sns.heatmap(
    cm, annot=False, cmap='Blues', ax=ax,
    xticklabels=INDEX_TO_WORD,
    yticklabels=INDEX_TO_WORD,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=8)
ax.set_title(
    f'LRCN — Confusion Matrix  (Val Acc: {best_val_acc:.2f}%)',
    fontsize=14
)
ax.set_ylabel('True sign'); ax.set_xlabel('Predicted sign')
plt.tight_layout()
path = str(FIGS_DIR / '03_lrcn_confusion.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

## Top confused sign pairs
Identifies the most common mistakes for qualitative analysis.

In [ ]:
mistakes = []
for i in range(NUM_CLASSES_ACTUAL):
    for j in range(NUM_CLASSES_ACTUAL):
        if i != j and cm[i][j] > 0:
            mistakes.append((cm[i][j], INDEX_TO_WORD[i], INDEX_TO_WORD[j]))

mistakes.sort(reverse=True)
print(f"{'Count':>6}  {'True sign':<20}  {'Predicted as':<20}")
print('─' * 52)
for count, true_word, pred_word in mistakes[:15]:
    print(f"{count:>6}  {true_word:<20}  {pred_word:<20}")

# Log Results

In [ ]:
row = {
    'experiment'       : '03_lrcn',
    'date'             : datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    'model'            : f'VGG16 frozen + LSTM (h={LSTM_HIDDEN}, layers={LSTM_LAYERS})',
    'temporal_modeling': True,
    'augmentation'     : False,
    'transfer_learning': 'ImageNet pretrained (frozen backbone)',
    'train_signers'    : '00-15',
    'val_signers'      : '16-19',
    'num_classes'      : NUM_CLASSES_ACTUAL,
    'num_frames'       : NUM_FRAMES,
    'batch_size'       : BATCH,
    'learning_rate'    : LEARNING_RATE,
    'dropout'          : DROPOUT,
    'cnn_baseline_acc' : 13.57,
    'best_val_acc'     : round(best_val_acc, 2),
    'delta_from_cnn'   : round(best_val_acc - 13.57, 2),
    'epochs_run'       : len(history['val_acc']),
    'notes'            : 'LRCN baseline. 32 frames. 67 classes. No aug, no fine-tuning.'
}

log_path = str(LOGS_DIR / '03_lrcn_log.csv')
write_header = not os.path.exists(log_path)
with open(log_path, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header: writer.writeheader()
    writer.writerow(row)

print('Logged:')
for k, v in row.items(): print(f'  {k}: {v}')
print(f'\nSaved to: {log_path}')

# Summary

In [ ]:
print('=' * 52)
print('  LRCN BASELINE COMPLETE')
print('=' * 52)
print(f"  Best val accuracy   : {best_val_acc:.2f}%")
print(f"  CNN baseline (nb02) : 13.57%")
print(f"  Delta               : {best_val_acc - 13.57:+.2f} pp")
print()
print("  Ablation table so far:")
print(f"  Simple CNN          : 13.57%  (notebook 02)")
print(f"  LRCN (this)         : {best_val_acc:.2f}%  (notebook 03)")
print(f"  LRCN + aug          :  TBD    (notebook 04 - Hakeemi)")
print(f"  LRCN + transfer     :  TBD    (notebook 05 - Nico)")
print(f"  LRCN + best combo   :  TBD    (notebook 06 - Hakeemi)")
print()
print(f"  Checkpoint : {CKPT_PATH}")
print(f"  Curves     : results/figures/03_lrcn_curves.png")
print(f"  Confusion  : results/figures/03_lrcn_confusion.png")
print(f"  Log        : results/logs/03_lrcn_log.csv")
print()
print('  --> Next: 04_augmentation.ipynb (Hakeemi)')
print('=' * 52)